# RetinaVessel AI — multi-seed replication v4a (seed 7)

This notebook independently fits the standard U-Net, clean lightweight U-Net and degradation-trained lightweight U-Net with training RNG seed 7. The audited image split remains frozen. After training, it executes the complete validation-only v3 robustness/topology matrix.

**Protocol boundary:** seed 7 was predeclared with seeds 42 and 123. Do not alter model settings based on seed-42 results, and do not inspect or evaluate the official FIVES test images. Enable a GPU accelerator.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/simlexx-k/retinavessel-ai.git"
EXPECTED_COMMIT = "224e115808a3330ee17516f0348ccd8fda9aa5ba"
REPO = Path("/kaggle/working/retinavessel-ai-v4-seed7-code")
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_commit == EXPECTED_COMMIT
os.chdir(REPO)
print("Pinned replication code:", actual_commit)

Cloning into '/kaggle/working/retinavessel-ai-v4-seed7-code'...
HEAD is now at 224e115 Fix backend imports in Kaggle v3 test cell


Pinned replication code: 224e115808a3330ee17516f0348ccd8fda9aa5ba


In [2]:
import torch

sys.path.insert(0, str(REPO / "src"))
from retinavessel.data import find_fives_root, load_manifest, samples_from_frozen_split

assert torch.cuda.is_available(), "Enable a GPU accelerator before running v4a"
DATA_MOUNT = Path("/kaggle/input/datasets/simonkosgei/fives-retinal-dataset")
assert DATA_MOUNT.exists(), f"Missing attached FIVES dataset: {DATA_MOUNT}"
FIVES_ROOT = find_fives_root(DATA_MOUNT)
SPLIT_MANIFEST = REPO / "splits/fives_seed42.json"
EXPECTED_SPLIT_SHA256 = "aee8809176974d6f880013d85a3ab4a405a43121d0cf31363fcc47dd75256c87"
assert hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest() == EXPECTED_SPLIT_SHA256
split = json.loads(SPLIT_MANIFEST.read_text())
train_samples, validation_samples = samples_from_frozen_split(load_manifest(FIVES_ROOT, "train"), split)
assert len(train_samples) == 504 and len(validation_samples) == 89
assert len(load_manifest(FIVES_ROOT, "test")) == 200  # count only; no pixels opened
print("PyTorch:", torch.__version__)
print("CUDA device:", torch.cuda.get_device_name(0))
print("Frozen split: 504 train / 89 validation / 7 excluded")
print("Official test images inspected or evaluated: 0")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)

PyTorch: 2.11.0+cu130
CUDA device: Tesla T4
Frozen split: 504 train / 89 validation / 7 excluded
Official test images inspected or evaluated: 0
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], returncode=0)

In [3]:
test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = os.pathsep.join([str(REPO / "src"), str(REPO / "backend")])
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, env=test_environment, check=True)

..............                                                           [100%]
14 passed in 9.68s


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)

In [4]:
MAX_EPOCHS = 30
PATIENCE = 5
SEED = 7
RUN_TIMES = {}
EXPERIMENTS = [
    {"name": "unet_seed7_es", "model": "unet", "degradation": False},
    {"name": "lite_unet_seed7_es", "model": "lite_unet", "degradation": False},
    {"name": "lite_robust_seed7_es", "model": "lite_unet", "degradation": True},
]

def run_experiment(spec):
    output = REPO / "outputs" / spec["name"]
    last_checkpoint = output / "last.pt"
    command = [
        sys.executable, "-m", "retinavessel.train",
        "--data", str(DATA_MOUNT), "--model", spec["model"],
        "--epochs", str(MAX_EPOCHS), "--patience", str(PATIENCE),
        "--batch-size", "2", "--patch-size", "384",
        "--patches-per-image", "2", "--overlap", "64",
        "--split-manifest", str(SPLIT_MANIFEST),
        "--seed", str(SEED), "--device", "cuda",
        "--out", str(output),
    ]
    if spec["degradation"]:
        command.append("--degradation")
    if last_checkpoint.exists():
        state = torch.load(last_checkpoint, map_location="cpu", weights_only=False)
        complete = int(state["epoch"]) >= MAX_EPOCHS or int(state["epochs_without_improvement"]) >= PATIENCE
        if complete:
            print(f"{spec['name']} already complete at epoch {state['epoch']}; skipping")
            return
        command.extend(["--resume", str(last_checkpoint)])
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(REPO / "src")
    started = time.perf_counter()
    subprocess.run(command, cwd=REPO, env=environment, check=True)
    RUN_TIMES[spec["name"]] = time.perf_counter() - started
    for required in ("best.pt", "last.pt", "history.json", "split_manifest.json"):
        assert (output / required).is_file(), f"Missing {output / required}"

def save_partial_snapshot(label):
    target = Path(f"/kaggle/working/retinavessel_v4_seed7_{label}")
    archive = Path(shutil.make_archive(str(target), "zip", root_dir=REPO, base_dir="outputs"))
    print("Partial recovery archive:", archive)

## E1 replication — standard U-Net, seed 7

In [5]:
run_experiment(EXPERIMENTS[0])
save_partial_snapshot("after_unet")

device=cuda, model=unet, params=1,964,097


{'epoch': 1, 'training_loss': 1.0113010713978419, 'validation_dice': 0.4630150488010275}


{'epoch': 2, 'training_loss': 0.6763691229834443, 'validation_dice': 0.5753968879186592}


{'epoch': 3, 'training_loss': 0.5466604895653233, 'validation_dice': 0.6023653991251379}


{'epoch': 4, 'training_loss': 0.5093918027918017, 'validation_dice': 0.6527665755748746}


{'epoch': 5, 'training_loss': 0.46976947589289575, 'validation_dice': 0.6405487782883315}


... (4 outputs left out)

{'epoch': 10, 'training_loss': 0.37452016169175745, 'validation_dice': 0.6945896121563053}


{'epoch': 11, 'training_loss': 0.3614684941454066, 'validation_dice': 0.7921526108594161}


{'epoch': 12, 'training_loss': 0.3520049768840037, 'validation_dice': 0.7722166464420879}


{'epoch': 13, 'training_loss': 0.33040031963693245, 'validation_dice': 0.6775805301389506}
early stopping at epoch 13; best validation Dice=0.795747
Partial recovery archive: /kaggle/working/retinavessel_v4_seed7_after_unet.zip


## E2 replication — clean lightweight U-Net, seed 7

In [6]:
run_experiment(EXPERIMENTS[1])
save_partial_snapshot("after_lite")

device=cuda, model=lite_unet, params=236,156


{'epoch': 1, 'training_loss': 0.9573796979846463, 'validation_dice': 0.6732655674042956}


{'epoch': 2, 'training_loss': 0.514484357414028, 'validation_dice': 0.7400031114168855}


{'epoch': 3, 'training_loss': 0.4051753017637465, 'validation_dice': 0.751321155996627}


{'epoch': 4, 'training_loss': 0.398154514753038, 'validation_dice': 0.7895624574061442}


{'epoch': 5, 'training_loss': 0.37372905015945435, 'validation_dice': 0.7804039900791541}


... (3 outputs left out)

{'epoch': 9, 'training_loss': 0.3323302416071769, 'validation_dice': 0.8075077629473462}


{'epoch': 10, 'training_loss': 0.3398179348765148, 'validation_dice': 0.8021104162581707}


{'epoch': 11, 'training_loss': 0.31917010861197637, 'validation_dice': 0.8198771545427541}


{'epoch': 12, 'training_loss': 0.32211111429783085, 'validation_dice': 0.8103114373594673}
early stopping at epoch 12; best validation Dice=0.828647
Partial recovery archive: /kaggle/working/retinavessel_v4_seed7_after_lite.zip


## E3 replication — degradation-trained lightweight U-Net, seed 7

In [7]:
run_experiment(EXPERIMENTS[2])
save_partial_snapshot("training_complete")

device=cuda, model=lite_unet, params=236,156


{'epoch': 1, 'training_loss': 1.0190500057642422, 'validation_dice': 0.6244213721717459}


{'epoch': 2, 'training_loss': 0.5426159225226868, 'validation_dice': 0.6797231227875384}


{'epoch': 3, 'training_loss': 0.4536222971916672, 'validation_dice': 0.7760976049836148}


{'epoch': 4, 'training_loss': 0.420411598883451, 'validation_dice': 0.801310519419885}


{'epoch': 5, 'training_loss': 0.4014131792540115, 'validation_dice': 0.7937374620500663}


... (8 outputs left out)

{'epoch': 14, 'training_loss': 0.3293359652161598, 'validation_dice': 0.8383751087503952}


{'epoch': 15, 'training_loss': 0.33144210200638524, 'validation_dice': 0.8316282519042657}


{'epoch': 16, 'training_loss': 0.32541306281373616, 'validation_dice': 0.8200188736407803}


{'epoch': 17, 'training_loss': 0.31866462406007545, 'validation_dice': 0.8276161854593791}
early stopping at epoch 17; best validation Dice=0.845353
Partial recovery archive: /kaggle/working/retinavessel_v4_seed7_training_complete.zip


In [8]:
import pandas as pd
from retinavessel.model import UNet, model_params

rows = []
for spec in EXPERIMENTS:
    output = REPO / "outputs" / spec["name"]
    history = json.loads((output / "history.json").read_text())
    checkpoint_path = output / "best.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    assert checkpoint["seed"] == SEED
    network = UNet(base=int(checkpoint["base"]), lite=checkpoint["architecture"] == "lite_unet")
    network.load_state_dict(checkpoint["model_state"], strict=True)
    rows.append({
        "experiment": spec["name"], "model": checkpoint["architecture"],
        "degradation_training": spec["degradation"], "seed": SEED,
        "parameters": model_params(network), "completed_epochs": history[-1]["epoch"],
        "best_epoch": checkpoint["epoch"], "best_validation_dice": checkpoint["validation_dice"],
        "checkpoint_sha256": hashlib.sha256(checkpoint_path.read_bytes()).hexdigest(),
        "wall_minutes_this_notebook": round(RUN_TIMES.get(spec["name"], float("nan")) / 60, 2),
    })
summary = pd.DataFrame(rows).sort_values("best_validation_dice", ascending=False)
display(summary)
SUMMARY_DIR = REPO / "outputs" / "v4_seed7_summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(SUMMARY_DIR / "validation_summary.csv", index=False)
run_manifest = {
    "training_code_commit": EXPECTED_COMMIT, "split_manifest_sha256": EXPECTED_SPLIT_SHA256,
    "pytorch_version": torch.__version__, "cuda_device": torch.cuda.get_device_name(0),
    "maximum_epochs": MAX_EPOCHS, "early_stopping_patience": PATIENCE,
    "seed": SEED, "official_test_evaluated": False,
    "results": json.loads(summary.to_json(orient="records")),
}
(SUMMARY_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))

,experiment,model,degradation_training,seed,parameters,completed_epochs,best_epoch,best_validation_dice,checkpoint_sha256,wall_minutes_this_notebook
2,lite_robust_seed7_es,lite_unet,True,7,236156,17,12,0.845353,8cbc8bd7410b51248b071b4f6ff0b9c72317c94385f762...,90.33
1,lite_unet_seed7_es,lite_unet,False,7,236156,12,7,0.828647,915061edaf8dcebc2050bceb0a34b4bbbb36203a026c11...,63.28
0,unet_seed7_es,unet,False,7,1964097,13,8,0.795747,69e0e8ddd2175cbc6f0e52a6275cc1fbe38f2f47ac9728...,87.94


1544

## Full seed-7 validation matrix

This is the same 3 × 9 × 89 validation-only matrix used for seed 42. It performs no model selection and never accesses official-test pixels.

In [9]:
VALIDATION_OUTPUT = REPO / "outputs" / "validation_v4_seed7"
evaluation_command = [
    sys.executable, "-m", "retinavessel.validation",
    "--data", str(DATA_MOUNT), "--split-manifest", str(SPLIT_MANIFEST),
    "--expected-split-sha256", EXPECTED_SPLIT_SHA256,
    "--model", f"unet_seed7={REPO / 'outputs/unet_seed7_es/best.pt'}",
    "--model", f"lite_unet_seed7={REPO / 'outputs/lite_unet_seed7_es/best.pt'}",
    "--model", f"lite_robust_seed7={REPO / 'outputs/lite_robust_seed7_es/best.pt'}",
    "--topology", "--device", "cuda", "--warmup-runs", "2",
    "--degradation-seed", "2026", "--bootstrap-seed", "2026",
    "--bootstrap-repetitions", "2000", "--out", str(VALIDATION_OUTPUT),
]
evaluation_environment = os.environ.copy()
evaluation_environment["PYTHONPATH"] = str(REPO / "src")
subprocess.run(evaluation_command, cwd=REPO, env=evaluation_environment, check=True)
report = json.loads((VALIDATION_OUTPUT / "validation_report.json").read_text())
per_image = pd.read_csv(VALIDATION_OUTPUT / "validation_per_image.csv")
assert report["official_test_evaluated"] is False
assert report["validation_image_count"] == 89 and report["debug_validation_limit"] is None
assert len(per_image) == 3 * 9 * 89
display(pd.read_csv(VALIDATION_OUTPUT / "validation_summary.csv"))

model=unet_seed7, condition=clean, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  processed 60/89
  processed 70/89
  processed 80/89
  processed 89/89
model=unet_seed7, condition=blur_mild, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  processed 60/89
  processed 70/89
  processed 80/89
  processed 89/89
model=unet_seed7, condition=blur_severe, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  processed 60/89
  processed 70/89
  processed 80/89
  processed 89/89
model=unet_seed7, condition=contrast_mild, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  processed 60/89
  processed 70/89
  processed 80/89
  processed 89/89
model=unet_seed7, condition=contrast_severe, images=89
  processed 10/89
  processed 20/89
  processed 30/89
  processed 40/89
  processed 50/89
  process

,model_id,condition,category,severity,image_count,parameters,parameter_bytes,mean_latency_ms,median_latency_ms,p95_latency_ms,...,cldice_ci95_lower,cldice_ci95_upper,mean_thin_centerline_recall,median_thin_centerline_recall,thin_centerline_recall_ci95_lower,thin_centerline_recall_ci95_upper,mean_skeleton_component_ratio,median_skeleton_component_ratio,skeleton_component_ratio_ci95_lower,skeleton_component_ratio_ci95_upper
0,unet_seed7,clean,none,clean,89,1964097,7856388,859.896818,859.340529,886.815597,...,0.769116,0.827026,0.177733,0.170582,0.156414,0.199894,23.400192,18.250000,18.994434,28.098891
1,unet_seed7,blur_mild,blur,mild,89,1964097,7856388,862.848720,862.378751,885.907719,...,0.768856,0.825807,0.179063,0.168006,0.156924,0.201282,23.465347,18.000000,19.170931,28.139289
2,unet_seed7,blur_severe,blur,severe,89,1964097,7856388,864.287156,865.389331,893.621844,...,0.759363,0.818167,0.179578,0.176640,0.156459,0.203018,24.051639,18.333333,19.905208,28.922219
3,unet_seed7,contrast_mild,contrast,mild,89,1964097,7856388,858.375450,856.852911,883.197655,...,0.739704,0.804410,0.159555,0.158988,0.138670,0.181429,24.407638,17.666667,20.029031,29.250078
4,unet_seed7,contrast_severe,contrast,severe,89,1964097,7856388,862.476726,864.339473,885.072680,...,0.655307,0.725340,0.128734,0.115328,0.111982,0.146888,31.385556,23.000000,25.557296,37.398711
5,unet_seed7,illumination_mild,illumination,mild,89,1964097,7856388,856.928170,854.016697,881.135646,...,0.731181,0.797516,0.152662,0.143564,0.132129,0.171342,20.972062,12.750000,16.688090,25.700035
6,unet_seed7,illumination_severe,illumination,severe,89,1964097,7856388,857.610892,857.000785,883.934702,...,0.638925,0.718953,0.093447,0.091724,0.080293,0.106040,18.778986,11.750000,14.939510,22.626183
7,unet_seed7,noise_mild,noise,mild,89,1964097,7856388,861.164503,859.182127,891.832599,...,0.720332,0.780855,0.096357,0.076144,0.083405,0.110739,21.780492,13.750000,17.875433,26.051951
8,unet_seed7,noise_severe,noise,severe,89,1964097,7856388,860.033992,861.038693,885.724500,...,0.571291,0.642527,0.045435,0.035644,0.038148,0.053780,22.530955,15.750000,18.574728,27.297580
9,lite_unet_seed7,clean,none,clean,89,236156,944624,454.297804,454.427881,477.432636,...,0.817848,0.859498,0.207373,0.201606,0.185653,0.228823,23.072646,16.000000,18.864693,27.700680


In [10]:
ARCHIVE_BASE = Path("/kaggle/working/retinavessel_v4_seed7")
archive = Path(shutil.make_archive(str(ARCHIVE_BASE), "zip", root_dir=REPO, base_dir="outputs"))
print(f"Final artifact archive: {archive} ({archive.stat().st_size / 1024**2:.1f} MiB)")
print("Save this notebook version and download the archive before starting seed 123.")

Final artifact archive: /kaggle/working/retinavessel_v4_seed7.zip (34.8 MiB)
Save this notebook version and download the archive before starting seed 123.


## Stop here

Do not open the official FIVES test partition. Download and verify the seed-7 archive, then execute the separately versioned seed-123 replication. Only after all three seeds are aggregated may the model configuration and final test policy be frozen.